# Coordinate Reference Systems (CRS)

Companion notebook for the **EcoGeo Tutor** tutorial: *Coordinate Reference Systems (CRS)*.

Covers geographic vs projected vs web CRS, reprojection, distance/area calculation,
and the three most common CRS mistakes with fixes.

**Data you'll need:** any polygon shapefile with recognizable place names
(the tutorial uses a Colombia admin boundary file, `colombia.shp`) — update the path below.


## Setup

In [ ]:
!pip install geopandas shapely -q


## 1. Geographic CRS — EPSG:4326 (WGS84)

Raw storage format: degrees, not metres. Never calculate area directly in this CRS.

In [ ]:
import geopandas as gpd

gdf = gpd.read_file("colombia.shp")
print(gdf.crs)
print(gdf.crs.axis_info)

print(gdf.geometry.iloc[0].centroid)
# POINT (-74.297 4.570) - coordinates are in DEGREES

# DO NOT calculate area in geographic CRS!
# gdf.geometry.area  <- returns degrees^2, meaningless


## 2. Projected CRS — EPSG:32618 (UTM Zone 18N)

Metres-based; accurate for area and distance calculations within its zone.

In [ ]:
gdf_utm = gdf.to_crs("EPSG:32618")
print(gdf_utm.crs)

gdf_utm["area_km2"] = gdf_utm.geometry.area / 1e6
print(gdf_utm[["name", "area_km2"]].head())

from shapely.geometry import Point
bogota   = gpd.GeoSeries([Point(-74.0, 4.7)], crs="EPSG:4326").to_crs("EPSG:32618")
medellin = gpd.GeoSeries([Point(-75.6, 6.2)], crs="EPSG:4326").to_crs("EPSG:32618")
dist_km  = bogota.distance(medellin).iloc[0] / 1000
print(f"Bogota to Medellin: {dist_km:.1f} km")


## 3. Web Mercator — EPSG:3857

For rendering map tiles only. Never use for area or distance analysis - severe distortion at high latitudes.

In [ ]:
gdf_webmercator = gdf.to_crs("EPSG:3857")

# WRONG - do not calculate area in EPSG:3857
# gdf_webmercator.geometry.area  <- heavily distorted

# CORRECT workflow: analyse in 4326/UTM, convert to 3857 only for display
import folium
m = folium.Map(location=[4.7, -74.1], zoom_start=6)
folium.GeoJson(gdf.to_crs("EPSG:4326")).add_to(m)
m.save("colombia_map.html")


## The three most common CRS mistakes

In [ ]:
# Mistake 1: Layer misalignment - mixing CRS without checking
roads  = gpd.read_file("roads.shp")     # e.g. EPSG:4326
basins = gpd.read_file("basins.shp")    # e.g. EPSG:32618

# WRONG:
# overlay = gpd.overlay(roads, basins)   # layers don't align

# RIGHT: always align CRS before overlay
basins_aligned = basins.to_crs(roads.crs)
overlay = gpd.overlay(roads, basins_aligned)


In [ ]:
# Mistake 2: Area calculated in degrees
parks = gpd.read_file("parks.shp")      # EPSG:4326

# WRONG:
# parks["area"] = parks.geometry.area    # returns degrees^2

# RIGHT: reproject before area/distance
parks_m = parks.to_crs("EPSG:32618")
parks_m["area_km2"] = parks_m.geometry.area / 1e6


In [ ]:
# Mistake 3: Unknown or missing CRS
old_data = gpd.read_file("old_data.shp")
print(old_data.crs)   # None - CRS is unknown

# Assign CRS (does NOT move coordinates) if known from metadata,
# THEN reproject if needed
old_data = old_data.set_crs("EPSG:4326")
old_data = old_data.to_crs("EPSG:32618")


## Common EPSG codes reference

| EPSG Code | Name | Type | Coverage | Best use |
|---|---|---|---|---|
| EPSG:4326 | WGS 84 | Geographic | Global | GPS, raw data, data exchange |
| EPSG:3857 | Web Mercator | Projected (spherical) | Global (web) | Web map tiles only |
| EPSG:32618 | UTM Zone 18N | Projected (UTM) | Colombia / Caribbean | Analysis in N Colombia |
| EPSG:32633 | UTM Zone 33N | Projected (UTM) | Italy (centre) | Analysis in Italy |
| EPSG:3035 | LAEA Europe | Projected (Equal-area) | Europe | EU statistical analysis |
| EPSG:9377 | MAGNA-SIRGAS / CTM-12 | Projected | Colombia | Official Colombian CRS |

Look up any EPSG code at [epsg.io](https://epsg.io).

---
*Companion notebook for the EcoGeo Tutor tutorial on rcafe.vercel.app*
